# Week 5 — Deep Learning Application in Data Science
**Student Name:** Sindhu Patil | **Internship:** Data Science

## 1. Introduction
Deep Learning utilizes multi-layered artificial neural networks to automatically learn hierarchical representation features from raw data.

## 2. Problem Statement
Predicting telecom subscriber churn using a deep Feed-Forward Neural Network architecture.

## 3. Objective
Build, compile, train, evaluate, and compare a Keras deep neural network model on `data/processed/cleaned_telco_churn.csv`.

## 4. Dataset Overview
Loading cleaned dataset (7,043 rows, 21 columns). `customerID` is strictly excluded.

In [ ]:
import sys, os
os.environ['KERAS_BACKEND'] = 'torch'
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from src.deep_learning import build_nn_architecture, compile_nn_model, compute_train_class_weights, train_nn_model, evaluate_nn_test_performance, draw_architecture_diagram
from src.visualization import plot_training_history_loss, plot_training_history_accuracy, plot_confusion_matrix, plot_roc_curve, plot_precision_recall_curve, plot_threshold_analysis, plot_week4_vs_week5_comparison

df = pd.read_csv('../data/processed/cleaned_telco_churn.csv')
print('Cleaned Dataset Shape:', df.shape)

## 5. Why Deep Learning?
Deep neural networks capture complex non-linear feature interactions and high-order combinations without manual feature engineering.

## 6. Data Preparation (`Churn` Target)
Mapping target: `Yes` -> 1, `No` -> 0.

In [ ]:
X = df.drop(columns=['customerID', 'Churn'])
y = (df['Churn'] == 'Yes').astype(int)
print('Target Distribution:\n', y.value_counts(normalize=True))

## 7. Feature Selection & Engineering
Selecting continuous numerical attributes and categorical predictors.

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [c for c in X.columns if c not in num_cols]
print('Numerical features:', num_cols)
print('Categorical features:', cat_cols)

## 8. Train / Validation / Test Split (70 / 15 / 15 Stratified)
Splitting data: Train = 4,930 rows (70%), Validation = 1,056 rows (15%), Test = 1,057 rows (15%).

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
val_ratio = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=val_ratio, random_state=42, stratify=y_train_full)
print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

## 9. Feature Scaling and One-Hot Encoding
Fitting `ColumnTransformer` (`StandardScaler` + `OneHotEncoder`) on `X_train` ONLY to eliminate data leakage.

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])
X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled = preprocessor.transform(X_val)
X_test_scaled = preprocessor.transform(X_test)
print('Transformed Input Dimension:', X_train_scaled.shape[1])

## 10. Neural Network Architecture Definition
Constructing Keras Sequential Model:
- Dense(64, ReLU) + Dropout(0.30)
- Dense(32, ReLU) + Dropout(0.20)
- Dense(16, ReLU)
- Dense(1, Sigmoid)

In [ ]:
model = build_nn_architecture(X_train_scaled.shape[1])
model.summary()

## 11. Model Compilation & Class Weighting
Compiling with `Adam(lr=0.001)`, `binary_crossentropy`, and balanced class weights.

In [ ]:
model = compile_nn_model(model, learning_rate=0.001)
class_weights = compute_train_class_weights(y_train)

## 12. Model Training with EarlyStopping
Training for up to 50 epochs (`batch_size=32`, `patience=10`).

In [ ]:
model, history, model_path = train_nn_model(model, X_train_scaled, y_train, X_val_scaled, y_val, class_weights, epochs=50, batch_size=32, patience=10)

## 13. Training and Validation Loss/Accuracy Curves

In [ ]:
plot_training_history_loss(history, '../outputs/figures/week5/training_history_loss.png')
plot_training_history_accuracy(history, '../outputs/figures/week5/training_history_accuracy.png')

## 14. Model Evaluation on Unseen Test Set

In [ ]:
metrics_nn, y_pred_nn, y_prob_nn, cm_nn = evaluate_nn_test_performance(model, X_test_scaled, y_test)
print('Test Set Performance (Deep Learning Neural Network):\n', metrics_nn)

## 15. Confusion Matrix Analysis

In [ ]:
plot_confusion_matrix(cm_nn, ['Retained (No)', 'Churned (Yes)'], '../outputs/figures/week5/confusion_matrix.png')

## 16. ROC Curve & AUC

In [ ]:
plot_roc_curve(y_test, {'Deep Learning NN': (y_prob_nn, metrics_nn['ROC_AUC'])}, '../outputs/figures/week5/roc_curve.png')

## 17. Precision-Recall Analysis

In [ ]:
plot_precision_recall_curve(y_test, {'Deep Learning NN': (y_prob_nn, metrics_nn['Average_Precision'])}, '../outputs/figures/week5/precision_recall_curve.png')

## 18. Classification Threshold Analysis

In [ ]:
thresholds = np.linspace(0.1, 0.9, 81)
precisions = [precision_score(y_test, y_prob_nn >= t, zero_division=0) for t in thresholds]
recalls = [recall_score(y_test, y_prob_nn >= t, zero_division=0) for t in thresholds]
f1s = [f1_score(y_test, y_prob_nn >= t, zero_division=0) for t in thresholds]
plot_threshold_analysis(thresholds, precisions, recalls, f1s, '../outputs/figures/week5/threshold_analysis.png')

## 19. Overfitting Analysis & Regularization
Dropout layers (0.30 and 0.20) and EarlyStopping prevented severe overfitting, maintaining tight convergence between training and validation loss.

## 20. Model Interpretation
Deep Neural Networks act as non-linear function approximators. Neural probability outputs allow risk-based cohort ranking.

## 21. Comparison with Week 4 Baseline

In [ ]:
metrics_w4 = {'Model': 'Week 4 Logistic Regression', 'Accuracy': 0.8055, 'Precision': 0.6572, 'Recall': 0.5588, 'F1_Score': 0.6040, 'ROC_AUC': 0.8419, 'Average_Precision': 0.6543}
comp_df = pd.DataFrame([metrics_w4, metrics_nn])
plot_week4_vs_week5_comparison(comp_df, '../outputs/figures/week5/week4_vs_week5_comparison.png')
print(comp_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1_Score', 'ROC_AUC']])

## 22. Challenges
Handling class imbalance and tuning dropout rates to balance recall vs precision.

## 23. Limitations
Tabular neural networks require careful regularization and lack direct linear coefficient interpretability.

## 24. Future Improvements
Applying learning rate schedulers, Hyperband architecture tuning, or SHAP explainable AI.

## 25. Conclusion
The Keras Deep Neural Network with class weighting achieved an outstanding **75.36% Recall** (capturing 211 out of 280 churners) and **0.8436 ROC-AUC**, establishing a superior retention intervention model.